# Notebook 09 — Screening-transfer validation, negative controls, and honest blend selection

**Research question.** Does the improvement from predicting screening responses represent genuine transferable biochemical information, or can a similar gain arise from random targets and second-stage modeling alone?

This is a **new, standalone, leak-resistant control experiment**. It uses RDKit 2D descriptors + Morgan count fingerprints to keep negative controls practical on Colab CPU. It does **not** claim to replicate the original CheMeleon-containing Notebook 08 exactly. The earlier ~0.540 three-model blend remains an **exploratory historical benchmark**, not an independently tested result.

The notebook trains matched: (i) direct LightGBM; (ii) cross-fitted real-screening stack; (iii) **permuted-screen-label** negative-control stack; and (iv) a **nested, inner-fold-selected blend of (i) and (ii)**. All scaffolds in outer validation are excluded from both stages and from every auxiliary-only screening molecule. Inner folds select blending weights **without consulting outer labels**. Model selection is limited to these two LightGBM methods; it does not retrain or honestly select weights for the historical Chemprop ensemble.

**Typical usage:** `FAST_MODE=True` first (one outer fold), then `False` for all five. The fast output is diagnostic only. Increase permutation replicates and repeat with new outer seeds for publication-grade confirmation.

## 0 · Install packages (Colab)

In [ ]:
%pip -q install "rdkit>=2023.9" "lightgbm>=4.3" "scikit-learn>=1.5" "scipy>=1.11" pandas matplotlib

## 1 · Configuration and imports

In [ ]:
from pathlib import Path
from urllib.request import urlopen
from rdkit import Chem, rdBase, DataStructs
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, brier_score_loss
from scipy.stats import spearmanr
import pandas as pd, numpy as np, lightgbm as lgb
import io, json, hashlib, zipfile, os, platform, time, warnings, random
import matplotlib.pyplot as plt
ISOFORMS=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
URL='https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/'

def read_csv_source(filename, override=None):
    b=Path(override).read_bytes() if override else urlopen(URL+filename,timeout=180).read()
    return pd.read_csv(io.BytesIO(b)), hashlib.sha256(b).hexdigest()

def identity(smiles):
    mol=Chem.MolFromSmiles(str(smiles)) if pd.notna(smiles) else None
    if mol is None: return None,None
    canon=Chem.MolToSmiles(mol,canonical=True,isomericSmiles=True)
    core=MurckoScaffold.GetScaffoldForMol(mol)
    scaffold=(Chem.MolToSmiles(core,canonical=True,isomericSmiles=False) if core.GetNumAtoms()
              else 'ACYCLIC_'+canon)
    return canon,scaffold

def assign_primary_folds(primary,n_splits=5):
    fold=np.full(len(primary),-1,dtype=int)
    for k,(_,val) in enumerate(GroupKFold(n_splits=n_splits).split(primary,groups=primary.scaffold_group)):
        fold[val]=k
    assert (fold>=0).all()
    assert not pd.DataFrame({'g':primary.scaffold_group,'f':fold}).groupby('g').f.nunique().gt(1).any()
    return fold

def read_zip_table(path,filename):
    with zipfile.ZipFile(path) as z:
        names=[n for n in z.namelist() if n==filename or n.endswith('/'+filename)]
        if len(names)!=1:raise ValueError(f'{filename}: ambiguous/missing in {path}: {names}')
        return pd.read_csv(io.BytesIO(z.read(names[0])))

def molecular_features(smiles, n_bits=512):
    molecules=[Chem.MolFromSmiles(s) for s in smiles]
    assert all(m is not None for m in molecules)
    desc=[]
    for m in molecules:
        with np.errstate(all='ignore'):
            desc.append([fn(m) for _,fn in Descriptors._descList])
    desc=np.asarray(desc,dtype=np.float32)
    desc[~np.isfinite(desc)]=np.nan
    generator=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=n_bits)
    fps=np.zeros((len(molecules),n_bits),dtype=np.float32)
    for i,m in enumerate(molecules):
        fp=generator.GetCountFingerprint(m)
        DataStructs.ConvertToNumpyArray(fp,fps[i])
    features=np.column_stack((desc,fps)).astype(np.float32)
    feature_names=[f'D_{i}_{name}' for i,(name,_) in enumerate(Descriptors._descList)]+[f'M_{i}' for i in range(n_bits)]
    return features,feature_names

def mae_per_iso(frame, pred_col):
    scores={}
    for iso,chunk in frame.groupby('isoform'):
        scores[iso]=float(np.mean(np.abs(chunk.y_true.to_numpy()-chunk[pred_col].to_numpy())))
    return scores

def macro_mae(frame,pred_col):
    vals=mae_per_iso(frame,pred_col)
    return float(np.mean([vals[k] for k in ISOFORMS if k in vals]))

def save_zip(out_dir, filename):
    path=Path(filename)
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(Path(out_dir).rglob('*')):
            if p.is_file() and '_cache' not in p.parts:
                z.write(p,Path(out_dir).name+'/'+str(p.relative_to(out_dir)))
    return path

SEED=2026
FAST_MODE=True
N_SPLITS=5
FOLDS_TO_RUN=[0] if FAST_MODE else list(range(N_SPLITS))
# Five inner folds for the paper would be stronger; start with 2 or 3 for CPU practicality.
N_INNER_BLEND=2 if FAST_MODE else 3
N_SCREEN_CROSSFIT=2 if FAST_MODE else 3
N_BITS=512
N_ESTIMATORS=55 if FAST_MODE else 120
# One shuffled-label replicate is a smoke test, NOT a calibrated permutation p-value.
N_SHUFFLE_REPEATS=1 if FAST_MODE else 2
BLEND_WEIGHTS=[0.0,0.25,0.50,0.75,1.0]  # weight on screen-stack
BOOTSTRAP_REPS=50 if FAST_MODE else 500
RUN_TAG='v1'  # bump this if changing experimental design
DATA_OVERRIDES={}  # {'direct':'/content/my_direct.csv','screen':'/content/my_screen.csv'}
# Optional matching audit for the original Notebook 08 outputs.
NOTEBOOK08_RESULTS_ZIP=None  # '/content/cyp_notebook_08_results.zip'
# Mount Google Drive and set a persistent folder to survive Colab interruptions.
# from google.colab import drive; drive.mount('/content/drive')
PERSISTENT_ROOT=None # e.g. '/content/drive/MyDrive/cyp_notebook09'
BASE=Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')
OUT=BASE/'cyp_notebook_09_results';OUT.mkdir(parents=True,exist_ok=True)
print('LightGBM',lgb.__version__,'RDKit',rdBase.rdkitVersion,'fast mode',FAST_MODE)

## 2 · Load and verify the source tables and compound-level scaffold folds

In [ ]:
direct,sha_direct=read_csv_source('cyp-challenge-TRAIN_inhibition.csv',DATA_OVERRIDES.get('direct'))
screen,sha_screen=read_csv_source('cyp-challenge-single-concentration-TRAIN.csv',DATA_OVERRIDES.get('screen'))
required={'SMILES'}|{f'{iso}_pIC50_direct_inhibition' for iso in ISOFORMS}
assert required.issubset(direct.columns), sorted(required-set(direct.columns))
assert {'SMILES','enzyme','log2fc_estimate'}.issubset(screen.columns)
primary=direct.copy().reset_index(drop=True)
ident=[identity(s) for s in primary.SMILES]
primary['canonical_smiles']=[x for x,g in ident]
primary['scaffold_group']=[g for x,g in ident]
assert primary.canonical_smiles.notna().all() and primary.canonical_smiles.is_unique
primary['fold']=assign_primary_folds(primary,N_SPLITS)
manifest=primary[['canonical_smiles','scaffold_group','fold']].copy()
manifest.to_csv(OUT/'fold_manifest.csv',index=False)
if NOTEBOOK08_RESULTS_ZIP:
    previous=read_zip_table(NOTEBOOK08_RESULTS_ZIP,'fold_manifest.csv')
    left=manifest.set_index('canonical_smiles').fold
    right=previous.set_index('canonical_smiles').fold
    assert set(left.index)==set(right.index)
    assert np.array_equal(left.to_numpy(),right.loc[left.index].to_numpy()), 'Previous fold mismatch'
    print('Verified identical fold manifest with Notebook 08')
# The screening data are LONG format, with 4 CYP rows per compound.
screen=screen.copy()
screen['canonical_smiles']=[identity(s)[0] for s in screen.SMILES]
screen=screen[screen.canonical_smiles.notna()].copy()
screen['enzyme']=screen.enzyme.astype(str).str.strip().str.upper()
screen=screen[screen.enzyme.isin(ISOFORMS)].copy()
screen['log2fc_estimate']=pd.to_numeric(screen.log2fc_estimate,errors='coerce')
screen=screen[screen.log2fc_estimate.notna()].copy()
# If batches duplicate the same compound/enzyme, collapse before fitting.
screen_wide=screen.pivot_table(index='canonical_smiles',columns='enzyme',values='log2fc_estimate',aggfunc='mean')
for iso in ISOFORMS:
    if iso not in screen_wide:raise ValueError(f'Missing screen endpoint {iso}')
all_smiles=list(primary.canonical_smiles)+sorted(set(screen_wide.index)-set(primary.canonical_smiles))
smiles_index={s:i for i,s in enumerate(all_smiles)}
all_groups=np.asarray([identity(s)[1] for s in all_smiles],dtype=object)
N_PRIMARY=len(primary)
Y=np.column_stack([pd.to_numeric(primary[f'{iso}_pIC50_direct_inhibition'],errors='coerce').to_numpy(float) for iso in ISOFORMS])
SC=np.full((len(all_smiles),4),np.nan,dtype=np.float32)
for smi,row in screen_wide.iterrows():
    idx=smiles_index[smi]
    SC[idx]=[row.get(iso,np.nan) for iso in ISOFORMS]
print('Direct molecules',N_PRIMARY,'screening compounds',len(screen_wide),'union',len(all_smiles))
print('Screen counts',dict(zip(ISOFORMS,np.isfinite(SC).sum(axis=0).tolist())))
assert len(set(primary.canonical_smiles))==len(primary)

## 3 · Featurize molecules once; cache raw descriptors and fingerprint counts

In [ ]:
config_id={'sha_direct':sha_direct,'sha_screen':sha_screen,'fold_sha':hashlib.sha256(manifest.to_csv(index=False).encode()).hexdigest(),
           'seed':SEED,'bits':N_BITS,'trees':N_ESTIMATORS,'inner_blend':N_INNER_BLEND,
           'inner_screen':N_SCREEN_CROSSFIT,'null_repeats':N_SHUFFLE_REPEATS,'weights':BLEND_WEIGHTS,'tag':RUN_TAG}
RUN_ID=hashlib.sha256(json.dumps(config_id,sort_keys=True).encode()).hexdigest()[:16]
CACHE=BASE/('cyp_nb09_cache_'+RUN_ID);CACHE.mkdir(parents=True,exist_ok=True)
feature_file=CACHE/'molecular_features.npz'
if feature_file.exists():
    X=np.load(feature_file)['X']
    assert X.shape[0]==len(all_smiles)
    print('Loaded cached molecular features',X.shape)
else:
    X,names=molecular_features(all_smiles,N_BITS)
    np.savez_compressed(feature_file,X=X)
    print('Cached',X.shape)
# No model or preprocessing is fit on all molecule labels. Descriptor calculation uses structures only.
assert X.shape[0]==len(all_smiles)

## 4 · Nested pipelines with explicit scaffold exclusions

For each outer fold, stage-one screening models must exclude its entire held-out scaffold set. For each **inner screen cross-fit** they must also exclude the inner held-out scaffolds. When selecting blend weights, a further nested validation is created **inside** the outer training compounds.

All three second-stage models (direct, real screen, permuted screen) use the same LightGBM hyperparameters, train compounds, and molecular features. The negative-control screen labels are shuffled *within CYP*, among eligible training molecules, with the exact same missing-label mask.

In [ ]:
def make_model(seed):
    return lgb.LGBMRegressor(n_estimators=N_ESTIMATORS,learning_rate=.045,num_leaves=15,
        min_child_samples=6 if FAST_MODE else 15,colsample_bytree=.85,
        reg_lambda=3.0,verbosity=-1,n_jobs=2,random_state=seed)

def impute_with_fit(fit_idx):
    # NaN medians are estimated only from available training structures, never validation structures.
    median=np.nanmedian(X[fit_idx],axis=0)
    median[~np.isfinite(median)]=0.0
    return np.where(np.isfinite(X),X,median[None,:]).astype(np.float32)

def screen_features_for_split(train_idx, eval_idx, forbidden_scaffolds, salt, shuffled=False):
    """Return out-of-fold screen predictions for train_idx and full-fit predictions for eval_idx."""
    tr_groups=all_groups[train_idx]; ev_groups=set(all_groups[eval_idx])
    assert not set(tr_groups)&ev_groups, 'Train/eval scaffold overlap'
    forbidden=set(forbidden_scaffolds)|ev_groups
    screen_eligible=np.flatnonzero(~np.isin(all_groups,list(forbidden)))
    assert set(all_groups[screen_eligible]).isdisjoint(forbidden)
    ncv=min(N_SCREEN_CROSSFIT,len(set(tr_groups)))
    if ncv<2:raise ValueError('Not enough distinct scaffolds for screen cross-fitting')
    cv=GroupKFold(n_splits=ncv)
    out_train=np.full((len(train_idx),4),np.nan,dtype=np.float32)
    out_eval=np.full((len(eval_idx),4),np.nan,dtype=np.float32)
    audit=[]
    # Impute anew for each inner-fitting pool; inner validation molecular features are excluded.
    def one_model(fit_indices,j,predict_idx,model_seed,xx):
        obs=fit_indices[np.isfinite(SC[fit_indices,j])]
        vals=SC[obs,j].copy()
        if shuffled and len(vals)>1:
            vals=np.random.default_rng(model_seed+31111).permutation(vals)
        if len(obs)<12 or np.nanstd(vals)<1e-8:
            return np.full(len(predict_idx),float(np.mean(vals)) if len(vals) else 0.0,dtype=np.float32),len(obs)
        model=make_model(model_seed)
        model.fit(xx[obs],vals)
        return model.predict(xx[predict_idx]).astype(np.float32),len(obs)
    for sub,(fit_pos,valid_pos) in enumerate(cv.split(train_idx,groups=tr_groups)):
        val_idx=train_idx[valid_pos]
        val_groups=set(all_groups[val_idx])
        allowed=screen_eligible[~np.isin(all_groups[screen_eligible],list(val_groups))]
        assert not set(all_groups[allowed])&val_groups
        xx_inner=impute_with_fit(allowed)
        for j in range(4):
            pred,nobs=one_model(allowed,j,val_idx,salt+sub*31+j*7,xx_inner)
            out_train[valid_pos,j]=pred
            audit.append({'stage':'crossfit','n_fit_screen':nobs,'inner_scaffold_overlap':0,
                          'outer_scaffold_overlap':0,'shuffled':shuffled})
    xx_outer=impute_with_fit(screen_eligible)
    for j in range(4):
        pred,nobs=one_model(screen_eligible,j,eval_idx,salt+901+j*7,xx_outer)
        out_eval[:,j]=pred
        audit.append({'stage':'fullfit','n_fit_screen':nobs,'inner_scaffold_overlap':0,
                      'outer_scaffold_overlap':0,'shuffled':shuffled})
    assert np.isfinite(out_train).all() and np.isfinite(out_eval).all()
    return out_train,out_eval,audit

def fit_pipeline(train_idx, eval_idx, forbidden_scaffolds, salt,
                 include_null=False, include_base=True):
    """Stage 2 uses only DIRECT pIC50 labels from train_idx."""
    tr_groups=set(all_groups[train_idx]);ev_groups=set(all_groups[eval_idx])
    assert tr_groups.isdisjoint(ev_groups)
    assert tr_groups.isdisjoint(set(forbidden_scaffolds))
    # Median imputation deliberately fits on eligible direct-training compounds only.
    x=impute_with_fit(train_idx)
    sr_train,sr_eval,screen_audit=screen_features_for_split(train_idx,eval_idx,forbidden_scaffolds,salt,False)
    xtr_real=np.column_stack((x[train_idx],sr_train))
    xev_real=np.column_stack((x[eval_idx],sr_eval))
    if include_null:
        sn_train,sn_eval,null_audit=screen_features_for_split(train_idx,eval_idx,forbidden_scaffolds,salt+9999,True)
        xtr_null=np.column_stack((x[train_idx],sn_train))
        xev_null=np.column_stack((x[eval_idx],sn_eval))
        screen_audit+=null_audit
    result={key:np.full((len(eval_idx),4),np.nan,dtype=np.float32)
            for key in (['direct'] if include_base else [])+['screen']+(['null'] if include_null else [])}
    for j,iso in enumerate(ISOFORMS):
        observed=np.isfinite(Y[train_idx,j]);idtr=train_idx[observed]
        if len(idtr)<15:raise ValueError(f'Too few labels for {iso}: {len(idtr)}')
        for name in result:
            xxtr,xxev=(x[train_idx],x[eval_idx]) if name=='direct' else (
                (xtr_real,xev_real) if name=='screen' else (xtr_null,xev_null))
            model=make_model(salt+100*j+{'direct':1,'screen':2,'null':3}[name])
            model.fit(xxtr[observed],Y[idtr,j])
            result[name][:,j]=model.predict(xxev)
    return result,screen_audit

# A strictly nested weight choice among (direct LightGBM, real-screen stack) only.
def choose_inner_alpha(outer_fold,outer_train,outer_heldout_groups):
    groups=all_groups[outer_train]
    ncv=min(N_INNER_BLEND,len(set(groups)))
    if ncv<2:raise ValueError('Not enough scaffolds to select blend weights')
    rows=[]
    for inner,(fit_pos,valid_pos) in enumerate(GroupKFold(n_splits=ncv).split(outer_train,groups=groups)):
        tr=outer_train[fit_pos];val=outer_train[valid_pos]
        preds,_=fit_pipeline(tr,val,outer_heldout_groups,SEED+outer_fold*12000+inner*850)
        for i,idx in enumerate(val):
            for j,iso in enumerate(ISOFORMS):
                if np.isfinite(Y[idx,j]):
                    rows.append({'isoform':iso,'true':Y[idx,j],
                                 'direct':preds['direct'][i,j],'screen':preds['screen'][i,j]})
    inner_df=pd.DataFrame(rows)
    scores=[]
    for a in BLEND_WEIGHTS:
        f=inner_df.copy()
        f['pred']=(1-a)*f.direct+a*f.screen
        iso_errors=f.groupby('isoform').apply(lambda x:float(np.mean(np.abs(x['true']-x.pred))),
                                             include_groups=False)
        score=float(np.mean([iso_errors[i] for i in ISOFORMS if i in iso_errors.index]))
        scores.append({'alpha':a,'inner_macro_mae':score})
    scores_df=pd.DataFrame(scores)
    alpha=float(scores_df.sort_values(['inner_macro_mae','alpha']).iloc[0].alpha)
    return alpha,scores_df

## 5 · Execute and cache outer-fold negative controls and nested selection

In [ ]:
rows=[];selection_rows=[];screen_audit_rows=[]
for fold in FOLDS_TO_RUN:
    p=CACHE/f'outer_fold_{fold}.csv'
    q=CACHE/f'blend_selection_fold_{fold}.csv'
    a=CACHE/f'screen_audit_fold_{fold}.csv'
    if p.exists() and q.exists() and a.exists():
        print('Resuming cached fold',fold)
        rows.append(pd.read_csv(p));selection_rows.append(pd.read_csv(q));screen_audit_rows.append(pd.read_csv(a))
        continue
    outer_eval=np.flatnonzero(primary.fold.to_numpy()==fold)
    outer_train=np.flatnonzero(primary.fold.to_numpy()!=fold)
    forbidden=set(primary.scaffold_group.iloc[outer_eval])
    assert not set(all_groups[outer_train])&forbidden
    alpha,selection=choose_inner_alpha(fold,outer_train,forbidden)
    prediction,audit=fit_pipeline(outer_train,outer_eval,forbidden,SEED+fold*100000,include_null=True)
    # Extra independent permutations use the same directly fitted LightGBM baseline.
    extras=[]
    for replicate in range(1,N_SHUFFLE_REPEATS):
        null_pred,_=fit_pipeline(outer_train,outer_eval,forbidden,
                                  SEED+fold*100000+replicate*90000,
                                  include_null=True,include_base=False)
        extras.append(null_pred['null'])
    rec=[]
    for i,idx in enumerate(outer_eval):
        for j,iso in enumerate(ISOFORMS):
            if not np.isfinite(Y[idx,j]):continue
            d={'canonical_smiles':all_smiles[idx],'scaffold_group':all_groups[idx],
               'fold':fold,'isoform':iso,'y_true':float(Y[idx,j]),
               'p_direct':float(prediction['direct'][i,j]),
               'p_screen_real':float(prediction['screen'][i,j]),
               'p_screen_shuffled':float(prediction['null'][i,j]),
               'p_nested_blend':float((1-alpha)*prediction['direct'][i,j]+alpha*prediction['screen'][i,j]),
               'nested_screen_weight':alpha}
            for rep,arr in enumerate(extras,start=2):d[f'p_screen_shuffled_{rep}']=float(arr[i,j])
            rec.append(d)
    fold_df=pd.DataFrame(rec)
    selection=selection.assign(fold=fold,chosen_alpha=alpha)
    aud=pd.DataFrame(audit).assign(fold=fold)
    # Write temp + atomic rename to make resumption safe after Colab interruptions.
    for df,path in [(fold_df,p),(selection,q),(aud,a)]:
        tmp=path.with_suffix('.tmp');df.to_csv(tmp,index=False);os.replace(tmp,path)
    rows.append(fold_df);selection_rows.append(selection);screen_audit_rows.append(aud)
    print('Fold',fold,'complete; selected screen weight',alpha,'rows',len(fold_df))
predictions=pd.concat(rows,ignore_index=True)
selection=pd.concat(selection_rows,ignore_index=True)
audit=pd.concat(screen_audit_rows,ignore_index=True)
assert not predictions.duplicated(['isoform','canonical_smiles']).any()
assert (audit.outer_scaffold_overlap==0).all() and (audit.inner_scaffold_overlap==0).all()
predictions.to_csv(OUT/'validation_predictions.csv',index=False)
selection.to_csv(OUT/'inner_blend_selection.csv',index=False)
audit.to_csv(OUT/'screening_leakage_audit.csv',index=False)
print('Validation examples:',len(predictions),'folds:',sorted(predictions.fold.unique()))

## 6 · Compare real screening versus shuffled screening on matched held-out molecules

In [ ]:
summary=[]
columns=(['p_direct','p_screen_real','p_screen_shuffled','p_nested_blend'] +
         [f'p_screen_shuffled_{i}' for i in range(2,N_SHUFFLE_REPEATS+1)])

for iso,sub in predictions.groupby('isoform'):
    for model in columns:
        summary.append({'isoform':iso,'model':model,'mae':float(np.mean(np.abs(sub.y_true-sub[model]))),
                        'n':len(sub)})
summary_df=pd.DataFrame(summary)
macro=summary_df.groupby('model').mae.mean().reset_index().sort_values('mae')
summary_df.to_csv(OUT/'per_isoform_results.csv',index=False)
macro.to_csv(OUT/'macro_results.csv',index=False)
display(macro)
display(summary_df.pivot(index='isoform',columns='model',values='mae'))
print('Selected blend weights by outer fold:')
display(selection[selection.alpha==selection.chosen_alpha].sort_values('fold'))

## 7 · Scaffold-block bootstrap for paired differences

The following intervals describe variation **conditional on the trained and selected models**. They do **not** cover prior architecture search, tuning on Notebooks 05–08, or retraining instability. A null permutation test would require several shuffled-label repetitions, each rerunning the entire nested pipeline; one shuffled control alone is not a p-value.

In [ ]:
def cluster_bootstrap_comparison(frame,model_a,model_b,repetitions=BOOTSTRAP_REPS):
    # Aggregate error + observation counts by scaffold and isoform ONCE. Use a multinomial
    # resampling of the clusters; much faster and more memory-stable than repeated concat.
    groups=frame.scaffold_group.astype(str).unique()
    group_id=pd.Categorical(frame.scaffold_group.astype(str),categories=groups).codes.astype(np.int64)
    iso_id=np.array([ISOFORMS.index(s) for s in frame.isoform])
    key=group_id*4+iso_id
    size=(len(groups),4)
    counts=np.bincount(key,minlength=len(groups)*4).reshape(size)
    err_a=np.bincount(key,weights=np.abs(frame.y_true-frame[model_a]),
                       minlength=len(groups)*4).reshape(size)
    err_b=np.bincount(key,weights=np.abs(frame.y_true-frame[model_b]),
                       minlength=len(groups)*4).reshape(size)
    observed=macro_mae(frame,model_a)-macro_mae(frame,model_b)
    rng=np.random.default_rng(SEED+181)
    draws=[]
    for k in range(repetitions):
        weights=rng.multinomial(len(groups),np.full(len(groups),1/len(groups)))
        n=weights@counts
        if not np.all(n>0):continue
        score_a=np.mean((weights@err_a)/n)
        score_b=np.mean((weights@err_b)/n)
        draws.append(float(score_a-score_b))
    return {'comparison':f'{model_b} vs {model_a}','mae_improvement':observed,
            'bootstrap_lower':float(np.quantile(draws,.025)) if draws else np.nan,
            'bootstrap_upper':float(np.quantile(draws,.975)) if draws else np.nan,
            'n_bootstrap':len(draws)}

checks=[]
for alt in ['p_screen_real','p_screen_shuffled','p_nested_blend']:
    checks.append(cluster_bootstrap_comparison(predictions,'p_direct',alt))
checks.append(cluster_bootstrap_comparison(predictions,'p_screen_shuffled','p_screen_real'))
bootstrap_summary=pd.DataFrame(checks)
bootstrap_summary.to_csv(OUT/'paired_scaffold_bootstrap.csv',index=False)
display(bootstrap_summary)

## 8 · Figures and optional previous-model reference

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4.5))
for model in ['p_direct','p_screen_real','p_screen_shuffled','p_nested_blend']:
    sub=summary_df[summary_df.model==model]
    axs[0].plot(sub.isoform,sub.mae,marker='o',label=model.replace('p_',''))
axs[0].set_ylabel('MAE (pIC50)');axs[0].set_title('Matched outer validation');axs[0].legend(fontsize=7)
axs[1].bar(bootstrap_summary.comparison,bootstrap_summary.mae_improvement)
axs[1].axhline(0,color='black',lw=.7);axs[1].set_title('Paired macro MAE improvement')
axs[1].tick_params(axis='x',rotation=65,labelsize=7)
plt.tight_layout();plt.savefig(OUT/'01_screening_controls.png',dpi=160);plt.show()
if NOTEBOOK08_RESULTS_ZIP:
    old=read_zip_table(NOTEBOOK08_RESULTS_ZIP,'predictions.csv')
    keep=old[old.model.isin(['Direct_LGBM_control','ScreenStack_LGBM'])].copy()
    # This is archival OOF reference, not an independent held-out test for Notebook 09.
    ref=keep.groupby(['model','isoform']).apply(lambda x:float(np.mean(np.abs(x.y_true-x.y_pred))),include_groups=False)
    display(ref.unstack('isoform'))

## 9 · Export and interpretation

In [ ]:
metadata={'notebook':'09_negative_controls_nested_screen_validation.ipynb',
          'data_sha256':{'direct':sha_direct,'screen':sha_screen},'run_id':RUN_ID,
          'completed_folds':sorted(predictions.fold.unique().astype(int).tolist()),
          'fast_mode':FAST_MODE,'n_screen_crossfit':N_SCREEN_CROSSFIT,
          'n_inner_blend':N_INNER_BLEND,'n_null_repeats':N_SHUFFLE_REPEATS,
          'n_bits':N_BITS,'trees':N_ESTIMATORS,
          'note':'This is an independent RDKit+Morgan mechanism-control study; it does not reproduce the historic CheMeleon ensemble.'}
(OUT/'metadata.json').write_text(json.dumps(metadata,indent=2))
archive=save_zip(OUT,'cyp_notebook_09_results.zip')
print('Saved:',archive,'bytes:',archive.stat().st_size)
print('Interpretation checklist: real screening > shuffled? nested selection > direct? robust on 5 folds?')

## Methodological limits and next steps

- **No inference-time observed screening measurements:** only model-predicted screening features enter stage two.
- **No validation-label leakage:** held-out scaffolds are removed from every first-stage and second-stage fitting set. Inner dev folds are protected from their screening-model fits.
- The **permuted-screen control** should perform like a noninformative auxiliary target, but random noise/regularization may help or hurt; report results rather than assuming it must lose.
- The nested **blend weight** is selected using internal validation of the two LightGBM options; the outer-fold result is an honest estimate for **this pipeline**. The earlier three-way CheMeleon blend remains historically tuned and **is not included in this nested selector**.
- One five-fold split still reflects only one scaffold-partition scheme; repeat with cluster splits or other seeds and include negative-control replicates for stronger inference.
- Model evaluation here uses **MAE and diagnostics, not the official MA-ST-RAE**. Do not compare directly with challenge leaderboard scores.